In [7]:
# GPU-accelerated resample to 10nm using PyTorch
import numpy as np
import tifffile
import torch
import torch.nn.functional as F
import os

IN_PATH  = "Mitolysosome_Contents.tif"
OUT_PATH = "Mitolysosome_Contents_10nm_isotropic_GPU.tif"

# spacings (z,y,x) nm
orig_spacing = np.array([70.0, 0.893, 0.893])
target_spacing = np.array([10.0, 10.0, 10.0])

# GPU device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == 'cpu':
    raise RuntimeError("CUDA GPU not available. Install GPU-compatible PyTorch and drivers before running this cell.")

# Load input stack (z,y,x)
stack = tifffile.imread(IN_PATH)
stack = np.asarray(stack)
z_in, y_in, x_in = stack.shape
print("Input shape:", stack.shape, "dtype:", stack.dtype)

# compute zoom factors
zoom = (orig_spacing / target_spacing).tolist()   # [7.0, 0.1, 0.1]
z_zoom, y_zoom, x_zoom = zoom
z_out = int(round(z_in * z_zoom))
y_out = int(round(y_in * y_zoom))
x_out = int(round(x_in * x_zoom))
print("Output approx shape (z,y,x):", (z_out, y_out, x_out))

# integer downsample factors for Y/X
ds_y = int(round(1.0 / y_zoom))
ds_x = int(round(1.0 / x_zoom))
print("Downsample factors (y,x):", ds_y, ds_x)
if ds_y <= 0 or ds_x <= 0:
    raise ValueError("Invalid downsample factor computed.")

# center-crop so Y/X are divisible by ds factor
y_keep = (y_out * ds_y)
x_keep = (x_out * ds_x)
if y_keep > y_in or x_keep > x_in:
    # fallback: floor multiple
    y_keep = (y_in // ds_y) * ds_y
    x_keep = (x_in // ds_x) * ds_x
    y_out = y_keep // ds_y
    x_out = x_keep // ds_x
    print("Adjusted to available size. New output (y,x) =", (y_out, x_out))

y_pad = y_in - y_keep
x_pad = x_in - x_keep
y0 = y_pad // 2
x0 = x_pad // 2
y1 = y0 + y_keep
x1 = x0 + x_keep
print(f"Cropping Y:{y0}:{y1}, X:{x0}:{x1}")
stack_c = stack[:, y0:y1, x0:x1].copy()
z_c, y_c, x_c = stack_c.shape
print("Cropped shape:", stack_c.shape)

# find labels (skip 0)
labels = [int(v) for v in np.unique(stack_c) if int(v) != 0]
if len(labels) == 0:
    raise RuntimeError("No non-zero labels found.")
print("Labels:", labels)

# allocate soft maps on GPU (labels, z_out, y_out, x_out) as float32
# If this is too large for GPU memory, we will stream label by label and keep soft maps on CPU.
est_size_bytes = len(labels) * z_out * y_out * x_out * 4
print("Estimated softmaps size (bytes):", est_size_bytes)
use_in_gpu = est_size_bytes < (torch.cuda.get_device_properties(device).total_memory * 0.6)  # cautious threshold

if use_in_gpu:
    print("Allocating soft maps on GPU.")
    soft = torch.zeros((len(labels), z_out, y_out, x_out), dtype=torch.float32, device=device)
else:
    print("Allocating soft maps on CPU (to avoid GPU OOM).")
    soft = np.zeros((len(labels), z_out, y_out, x_out), dtype=np.float32)  # CPU array

# process each label
for i, lab in enumerate(labels):
    print(f"Processing label {lab} ({i+1}/{len(labels)})")
    binary = (stack_c == lab).astype(np.float32)   # cpu numpy (z,y,x)
    # convert to torch tensor and send to GPU
    t = torch.from_numpy(binary).to(device=device)            # shape (z,y,x)
    t = t.unsqueeze(0).unsqueeze(0)  # shape (1,1,z,y,x) => (N,C,D,H,W)
    # Upsample in Z by z_zoom (keep y/x same)
    # Torch interpolate accepts scale_factor for 3D; we set (z_zoom,1.0,1.0)
    with torch.cuda.amp.autocast(enabled=False):
        up = F.interpolate(t, scale_factor=(z_zoom, 1.0, 1.0), mode='trilinear', align_corners=False, recompute_scale_factor=True)
    # up shape: (1,1,z_out, y_keep, x_keep) maybe off by 1 -> correct if needed
    up = up.squeeze(0).squeeze(0)  # (z_out, y_keep, x_keep)
    # ensure z dimension matches z_out (pad/crop)
    if up.shape[0] != z_out:
        if up.shape[0] > z_out:
            up = up[:z_out, :, :]
        else:
            padz = z_out - up.shape[0]
            up = F.pad(up, (0,0,0,0,0,padz), mode='constant', value=0)  # pad on dim0
    # Now do block-average in XY via AvgPool2d applied per Z slice.
    # Reshape to (z_out, 1, y_keep, x_keep) -> treat each z as batch for 2D avgpool
    up2d = up.unsqueeze(1)  # shape (z_out, 1, y_keep, x_keep)
    # avgpool expects (N,C,H,W); apply kernel=ds_y x ds_x, stride same
    pool = torch.nn.AvgPool2d(kernel_size=(ds_y, ds_x), stride=(ds_y, ds_x), ceil_mode=False, count_include_pad=False)
    # Because pool works on 4D, we can collapse z_out into batch dimension
    bz, c, hy, wx = up2d.shape
    up2d_flat = up2d.reshape(bz * c, hy, wx).unsqueeze(1)  # (bz*c,1,hy,wx)
    pooled = pool(up2d_flat)  # (bz*c,1,y_out,x_out)
    pooled = pooled.squeeze(1).reshape(bz, c, pooled.shape[-2], pooled.shape[-1])  # (z_out,1,y_out,x_out)
    pooled = pooled.squeeze(1)  # (z_out, y_out, x_out)
    # store soft map
    if use_in_gpu:
        soft[i] = pooled
    else:
        soft[i, :, :, :] = pooled.cpu().numpy()
    # free temporary tensors
    del t, up, up2d, up2d_flat, pooled
    torch.cuda.empty_cache()

# Combine soft maps -> argmax
print("Combining softmaps...")
if use_in_gpu:
    # soft is GPU tensor
    # compute argmax across labels -> (z_out,y_out,x_out)
    stacked = soft  # (L, z, y, x)
    max_vals, idx = torch.max(stacked, dim=0)  # max_vals: (z,y,x), idx: (z,y,x)
    # map idx to labels
    idx_cpu = idx.cpu().numpy().astype(np.int32)
    max_vals_cpu = max_vals.cpu().numpy()
else:
    # soft is numpy (L,z,y,x)
    idx_cpu = np.argmax(soft, axis=0)   # (z,y,x)
    max_vals_cpu = np.max(soft, axis=0)

# build output array and map indices to label intensities
out = np.zeros((z_out, y_out, x_out), dtype=stack_c.dtype)
for i, lab in enumerate(labels):
    out[idx_cpu == i] = lab
# set background where max value below threshold (optional)
BG_CONF_THRESHOLD = 0.1
out[max_vals_cpu < BG_CONF_THRESHOLD] = 0

# Save output TIFF
tifffile.imwrite(OUT_PATH, out, photometric='minisblack', bigtiff=True)
print("Saved:", OUT_PATH)


Using device: cuda
Input shape: (22, 4092, 3222) dtype: uint32
Output approx shape (z,y,x): (154, 365, 288)
Downsample factors (y,x): 11 11
Cropping Y:38:4053, X:27:3195
Cropped shape: (22, 4015, 3168)
Labels: [1, 2, 3, 4, 5, 6, 7, 10, 11, 12, 20, 22, 23, 24, 25, 26, 27, 28, 31, 32, 33, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45]
Estimated softmaps size (bytes): 2072125440
Allocating soft maps on GPU.
Processing label 1 (1/32)


C:\Users\narendradp\AppData\Local\Temp\ipykernel_28152\894978614.py:92: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


Processing label 2 (2/32)
Processing label 3 (3/32)
Processing label 4 (4/32)
Processing label 5 (5/32)
Processing label 6 (6/32)
Processing label 7 (7/32)
Processing label 10 (8/32)
Processing label 11 (9/32)
Processing label 12 (10/32)
Processing label 20 (11/32)
Processing label 22 (12/32)
Processing label 23 (13/32)
Processing label 24 (14/32)
Processing label 25 (15/32)
Processing label 26 (16/32)
Processing label 27 (17/32)
Processing label 28 (18/32)
Processing label 31 (19/32)
Processing label 32 (20/32)
Processing label 33 (21/32)
Processing label 34 (22/32)
Processing label 35 (23/32)
Processing label 36 (24/32)
Processing label 38 (25/32)
Processing label 39 (26/32)
Processing label 40 (27/32)
Processing label 41 (28/32)
Processing label 42 (29/32)
Processing label 43 (30/32)
Processing label 44 (31/32)
Processing label 45 (32/32)
Combining softmaps...
Saved: Mitolysosome_Contents_10nm_isotropic_GPU.tif
